# RT-DETR on DocLayNet - Kaggle training run

Before running anything:
1. **Settings -> Accelerator -> GPU T4 x2**
2. **Settings -> Internet -> On**

Then use **Save Version -> Save & Run All (Commit)**, not interactive
run - keeps going even if the browser tab closes.

## 1. Confirm hardware

Kaggle doesn't always give you the accelerator you picked. This also
gets written to run_metadata.json for the reproducibility claim.

In [ ]:
!nvidia-smi

## 2. Install

Pinning ultralytics since RT-DETR's training args have moved between
minor versions.

In [ ]:
!pip install -q ultralytics==8.3.40 "datasets<4.0.0"


## 3. Pull the repo

Everything below calls the repo's scripts, not notebook-redefined logic -
keeps the notebook and repo from disagreeing on how data was prepared.

In [ ]:
REPO_URL = "https://github.com/RISHIVELS/rap-doclayout.git"

# %cd into a directory this cell is about to rm -rf breaks the shell's
# own working directory on the *next* run of this cell (getcwd fails
# because the path it thinks it is standing in no longer exists). So I
# step out to a stable parent directory before deleting anything.
%cd /kaggle/working
!rm -rf /kaggle/working/repo
!git clone -q $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1


## 4. Build the dataset

Downloads ~3.8GB on first run, converts to the image/label layout
Ultralytics expects. `--verify 12` renders sample pages to check next.

In [ ]:
!python scripts/prepare_dataset.py --out /kaggle/working/data/doclaynet --verify 12

## 5. STOP - check the labels before training

My class ordering (0-indexed alphabetical) is inferred, not confirmed.
If it's off by one, `Table` becomes `Section-header` everywhere and
nothing breaks - training runs fine, metrics look plausible, and every
result afterwards is describing the wrong thing.

No test catches this. Only look at the boxes: is `Table` drawn around
a table? Is `Picture` around a figure? Fix before spending GPU quota
if not.

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image

checks = sorted(Path("/kaggle/working/data/doclaynet/label_check").glob("*.png"))
print(f"{len(checks)} pages to check\n")

for path in checks[:4]:
    display(Image.open(path).resize((620, 620)))

## 6. Train

RT-DETR-L, 30 epochs, batch 8 at 640px, AMP on.

640px is a compromise for the T4's memory/time budget - thin classes
like Footnote/Page-footer lose detail at this resolution, noted as a
known limitation rather than hidden.

Checkpointing every epoch so a dropped session costs one epoch, not
the run.

In [ ]:
!python scripts/train.py \
    --data /kaggle/working/data/doclaynet/doclaynet.yaml \
    --epochs 30 \
    --batch 8 \
    --imgsz 640 \
    --device 0 \
    --name rtdetr_doclaynet

## 7. Evaluate

Overall mAP, per-class AP, confusion matrix, per-category breakdown,
query-saturation rate. Per-category matters most - one aggregate mAP
can't tell you if the model learned structure or just learned what
financial reports look like.

In [ ]:
!python scripts/evaluate.py \
    --weights runs/detect/rtdetr_doclaynet/weights/best.pt \
    --data /kaggle/working/data/doclaynet/doclaynet.yaml \
    --out reports

## 8. Mine failure cases

Ranks test images by error, renders the worst side by side with ground
truth. The memo's failure cases come from here, not from guessing.

In [ ]:
!python scripts/mine_failures.py \
    --weights runs/detect/rtdetr_doclaynet/weights/best.pt \
    --data /kaggle/working/data/doclaynet \
    --out reports/failures \
    --top 25

## 9. Collect outputs

Weights, metrics, run metadata, failure renders.

In [ ]:
import shutil
from pathlib import Path

run = Path("runs/detect/rtdetr_doclaynet")
out = Path("/kaggle/working/submission")
out.mkdir(exist_ok=True)

shutil.copy(run / "weights/best.pt", out / "best.pt")
shutil.copy(run / "run_metadata.json", out / "run_metadata.json")
if Path("reports").exists():
    shutil.copytree("reports", out / "reports", dirs_exist_ok=True)

for path in sorted(out.rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size / 1e6:8.1f} MB  {path.relative_to(out)}")